<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_95_image_captioning_adversarial_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🔤 **Module 95 — Image Captioning + Adversarial NLP** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 13 · Vision Robustness & Generative Translation · **FINAL MODULE**


# Module 95 — Image Captioning + Adversarial NLP

> ndb796 closed the practice repo with two seemingly distant notebooks — **Neural Image Captioning** (Show-and-Tell with a ResNet-101 encoder + LSTM decoder) and **TextFooler** (a black-box adversarial attack on BERT). This module ties them together because they are the **two ends** of the multimodal-language pipeline that powers modern AI: *understanding* (encode an image into text) and *robustness* (attack a text classifier). The two ends share an architecture — both rely on token-level autoregressive decoding over a learned feature space — and modern systems (LLaVA, GPT-4V, Claude 3.7-Vision, Gemini-2.5-Pro) fail in *both* ways.

This is the **last module of the 95-module course**. After it: nothing scheduled — just the field, your projects, and time.

### What you'll cover
1. The **Show-and-Tell** captioning recipe (Vinyals 2015 → 2026)
2. **Encoder choice** — VGG → ResNet-101 → ViT → CLIP → DINOv2 → SigLIP
3. **Decoder choice** — LSTM → Transformer → frozen-LLM cross-attention
4. **Attention** in captioning — Soft + Hard (Xu 2015), then full Transformer
5. **Beam search · top-p · CIDEr-optimization** for caption decoding
6. The modern picker — **BLIP-2 · LLaVA · LLaVA-NeXT · InternVL · Florence-2 · Qwen-VL · MiniCPM-V**
7. **TextFooler revisited** — the black-box word-swap attack on BERT (M91 bridge)
8. **NLP adversarial canon** — HotFlip · BERT-Attack · GBDA · TextAttack framework
9. **Caption-level adversarial** — attack the **captioner**, not the classifier
10. **The 95-module wrap-up + course epilogue**


## 1 · Show-and-Tell — the 2015 ancestor of every VLM

**Show and Tell** (Vinyals, Toshev, Bengio, Erhan, CVPR 2015) was the paper that married vision and language end-to-end. The architecture is two boxes:

```
image  ─►  CNN encoder  ─►  v  (single feature vector, e.g. 2048-dim)
                            │
                            ▼
                       [BOS]  v ─► LSTM ─► w_1 ─► LSTM ─► w_2 ─► ... ─► [EOS]
```

The image feature `v` is treated as **the first input** to a language-model LSTM. The LSTM autoregressively generates one word at a time. Trained with maximum-likelihood on (image, caption) pairs from **MS-COCO** (~120k images, 5 captions each). Tested on the same.

**The intellectual move**: an LSTM language model, conditioned on an image embedding, learns to *describe* what's in the image — without explicit object detection, scene parsing, or relationship reasoning. The same architecture, scaled up with attention and replaced with transformers + frozen LLMs, becomes **every modern vision-language model** (LLaVA, Gemini-Vision, GPT-4V, BLIP-2).

> 📜 **The lineage.** Show-and-Tell (2015, LSTM) → Show-Attend-Tell (2015, soft attention) → Up-Down (2018, region features) → Transformer captioner (2019) → OSCAR / VinVL (2020, contrastive) → BLIP / BLIP-2 (2022, Q-Former + frozen LLM) → LLaVA (2023, projector + frozen LLM) → GPT-4V / Claude Vision / Gemini-Vision (2024+, frontier VLMs). **Every box on that path is the same recipe with more knowledge moved into the encoder + decoder.**


## 2 · Encoder choice — what "seeing" looks like

| Era | Encoder | Output | Why |
|---|---|---|---|
| 2014-2015 | **VGG-16/19** | global 4096-d FC feature | First end-to-end visual feature |
| 2015-2018 | **ResNet-50/101/152** | global 2048-d pooled feature | ndb796's notebook; deeper + skip connections + faster |
| 2018-2020 | **Region features** (Faster R-CNN on Visual Genome) | 36 box × 2048-d | Up-Down attention: explicit objects |
| 2020-2022 | **ViT** (Dosovitskiy 2020) | 196 patch × 768-d (or N×D) | Self-attention from pixels; uniform interface |
| 2021-2023 | **CLIP-ViT** (Radford 2021) | 768/1024-d pooled + grid | Text-alignment baked in via 400M contrastive pairs |
| 2023+ | **SigLIP / DINOv2 / SAM-encoder** | better features, lower dim | Cleaner gradients, less hallucination, robust to crops |
| 2024+ | **Native multimodal** (Gemini-1.5/2, GPT-4o) | proprietary mixed-modal features | Trained jointly with text from day 0 |

**Practical rule (2026).** For *captioning*, the encoder choice matters less than the projector quality (Section 3). Use **CLIP-ViT-L/14** or **SigLIP-So400m** if you're building from scratch; use **the existing encoder of a frontier VLM** if you're fine-tuning on top.

> 🧐 **Why ndb796 chose ResNet-101.** It was the right balance in 2017: deeper than ResNet-50, fast enough to fine-tune end-to-end on a single GPU, pretrained ImageNet features that *transfer cleanly* to MS-COCO captioning. In 2026, you'd use CLIP-ViT-B/16 for the same job — same compute, much stronger features, free language alignment.


## 3 · Decoder choice — from LSTM to "frozen LLM + projector"

The decoder side of the captioner is where the 2015 → 2026 architecture moved the most:

| Decoder | When | Notes |
|---|---|---|
| **LSTM** (Vinyals 2015) | 2015-2017 | Image vector = initial hidden state OR first input |
| **LSTM + Soft Attention** (Xu 2015 "Show, Attend, Tell") | 2015-2018 | Decoder attends to spatial encoder cells |
| **LSTM + Hard Attention** | 2015-2017 | REINFORCE-trained discrete attention — slower, marginal gain |
| **Transformer decoder** (Cornia 2020 "M²" Meshed-Memory) | 2019-2021 | Standard Tr-decoder cross-attending image patches |
| **BLIP / BLIP-2 Q-Former** (Li 2022, 2023) | 2022-2024 | 32 learned queries pull from image; project to frozen LLM |
| **Linear projector + frozen LLM** (LLaVA-1.5 / 1.6) | 2023-2025 | 2-MLP projector; image tokens prepended to LLM input |
| **Native mixed-token decoder** (GPT-4o / Gemini-2) | 2024+ | Image and text tokens go through the same transformer stack |

**LLaVA's whole trick** is one of the most elegant in 2023: take a frozen LLM (Vicuna-7B), a frozen CLIP-ViT encoder, and a tiny 2-layer MLP **projector** that maps CLIP image patches into the LLM's token embedding space. *That's it.* Train only the projector on 558k (image, caption) pairs + 158k (image, instruction, response) pairs. **2 GPU-days on 8×A100.** Hits 85% of GPT-4V performance on most benchmarks at 1% of the parameter count.

> 🎯 **The 2026 lesson.** Captioning is no longer a stand-alone problem — it's a sub-capability of a multimodal LLM. If you need captions for a product, you don't train a captioner; you call **GPT-4V / Claude-3.7-Vision / Gemini-2.5-Pro** for quality, or **LLaVA-NeXT-34B / InternVL-2 / Qwen2-VL-7B / MiniCPM-V-2.6** for self-hosted (M90 callback). Building a captioner from scratch is now a **pedagogical exercise**, not a production exercise.


## 4 · Soft attention — the bridge between 2015 and transformers

Soft attention in captioning (Xu et al. 2015, "Show, Attend, and Tell") is **the same attention mechanism** later popularized by Transformers — just framed as a one-direction cross-attention from decoder hidden state to encoder feature map.

At each decoder step `t`:
```
e_{t, i} = f_att(h_{t-1}, a_i)            # MLP scoring of decoder state vs each encoder location i
α_{t, i} = softmax_i(e_{t, i})            # attention weights over spatial locations
c_t     = Σ_i α_{t, i} · a_i              # context vector — weighted feature
h_t     = LSTM(h_{t-1}, [w_{t-1}, c_t])   # next decoder step
w_t     = argmax softmax(W·h_t)           # output word
```

**This is exactly cross-attention with a single head, before Vaswani et al. named it.** Modern Transformer captioners are just multi-head, multi-layer versions of the same equation. Knowing this makes M19/M20 (transformer attention) and M84 (RNN/LSTM/GRU) feel like one continuous derivation rather than two separate technologies.

> 🔄 **The intellectual continuity.** Soft attention (2015) → Bahdanau seq2seq attention (2014, M84) → Vaswani Transformer (2017, M19/M20) → cross-attention in Stable Diffusion (M86) → cross-attention in Q-Former (BLIP-2) → cross-attention in LLaVA (image tokens → text tokens). **Cross-attention is one of the most reusable building blocks in deep learning.** It is the thing that connects the entire course.


## 5 · Decoding strategies — beam search, top-p, CIDEr-optim

A captioner emits a distribution over the next word. Decoding strategy turns that distribution into a sentence. The choice is non-trivial for caption quality:

| Strategy | Idea | When |
|---|---|---|
| **Greedy** | `argmax` each step | Fast, hurts diversity |
| **Beam search** (B=3-5) | Keep top-B partial sequences | **The standard for captioning** |
| **Sampling** | Random draw from distribution | Diversity, less coherent |
| **Top-k** (k=40) | Sample from k highest-prob | Cuts garbage tail |
| **Top-p / nucleus** (p=0.9, Holtzman 2019) | Sample from smallest set s.t. Σp ≥ 0.9 | Default for general LMs, less so for captioning |
| **Temperature** (T∈[0.5, 1.5]) | Sharpen / flatten the distribution | Tune diversity |
| **CIDEr / BLEU-RL** (Rennie 2017 "SCST") | Train with REINFORCE on the eval metric directly | +2-3 CIDEr over MLE training |

**Beam search is the dominant 2026 choice for captioning** (vs sampling for general text), because captions reward **conservative coherence** more than diversity — a missed object hurts more than a generic phrase.

**Self-Critical Sequence Training (SCST, Rennie 2017)** flips the loss to optimize the eval metric directly:

```
L = − E_{w∼p_θ}[ (CIDEr(w) − CIDEr(w_greedy)) · log p_θ(w) ]
```

The greedy baseline is subtracted so REINFORCE has lower variance. This was the *standard* COCO-captioning recipe from 2018-2022, worth 2-3 CIDEr points.

> ⚖️ **A subtle modern issue.** CIDEr-optimized models *score* better but produce more **generic** captions ("a person standing in front of a building"). For real-world use (alt-text, accessibility, retrieval), MLE-trained or RLHF-aligned captioners often feel **more useful** despite scoring lower. The lesson reappears in M27 / M70 eval — leaderboard metrics ≠ user value.


## 6 · The 2026 modern picker

What you'd actually ship today instead of training Show-and-Tell from scratch:

| Tier | Closed-source | Open-source |
|---|---|---|
| **Frontier** | GPT-4o / Claude-3.7-Sonnet-Vision / Gemini-2.5-Pro / GPT-4.5 | Llama-3.2-90B-Vision · InternVL-2.5-78B · Qwen2-VL-72B |
| **Mid-tier** | Claude-3.5-Haiku / GPT-4o-mini | LLaVA-NeXT-34B · InternVL-2-26B · MiniCPM-V-2.6 (8B) |
| **Small / on-device (M90 callback)** | Gemini Nano-3 · Apple AFM-Vision | Phi-3.5-Vision · Llama-3.2-11B-Vision · MiniCPM-V-2.0-3B · Florence-2 (0.7B / 1.5B) |
| **Specialized** | — | CogVLM-2 (best at OCR + grounding) · SAM-2 (segmentation) · GroundingDINO (open-vocab detection) |

**Architecture summary** (May 2026): all of these use the same recipe — frozen / lightly-tuned vision encoder (CLIP / SigLIP / DINOv2 variant) → projector or Q-Former → LLM. They differ in (a) which LLM is frozen, (b) what fine-tune data is used, (c) what tokens (multi-resolution image, video frames, audio) the encoder produces.

**Code in 5 lines.** Today, "build a captioner" is:
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
m = AutoModelForCausalLM.from_pretrained("openbmb/MiniCPM-V-2_6", trust_remote_code=True)
t = AutoTokenizer.from_pretrained("openbmb/MiniCPM-V-2_6")
msgs = [{"role": "user", "content": [image, "Describe this image."]}]
print(m.chat(image=image, msgs=msgs, tokenizer=t)["text"])
```

That's it. No CNN. No LSTM. No beam search to write. The captioner is a multimodal LLM call.

> 🪞 **A historical note.** ndb796's Neural Image Captioning notebook (~2018-2019) is ~200 lines of PyTorch. The 2026 equivalent above is ~5 lines and produces dramatically better captions. The 2015-2024 progress in vision-language is largely **infrastructure progress**: bigger encoders, bigger LMs, more data, better connectors. The intellectual core (cross-attention + autoregressive decoder) is unchanged.


## 7 · TextFooler revisited — the M91 bridge

We covered TextFooler briefly in M91. Here we go deeper because it is the prototype of every adversarial NLP attack — and the conceptual ancestor of every LLM jailbreak (M74).

**Threat model.** Black-box: attacker only sees the target's top-class predictions or probabilities, not the model's gradients. Constraint: the swapped word must **preserve meaning** to a human — measured via:
- **Counter-fitted GloVe synonyms** (Mrkšić 2016) — embeddings where synonyms cluster and antonyms repel
- **Universal Sentence Encoder similarity ≥ 0.7** (Cer 2018) — sentence-level meaning preservation
- **Part-of-speech preservation** — swap nouns for nouns, verbs for verbs
- **Stop-word skip** — never swap "the", "is", "a"

**Algorithm:**
```
1. For each word w_i in input x:
     score_i = P(y_true | x) − P(y_true | x with w_i masked)
2. Sort words by descending score_i
3. For each high-impact word w_i:
     a. Get candidate synonyms via counter-fitted GloVe (top K=50)
     b. Filter: keep only those with same POS + USE-similarity ≥ 0.7
     c. For each surviving candidate:
        - replace w_i with candidate
        - query target → if class flipped, return adversarial example
        - else track the candidate that lowers P(y_true) most
     d. If no flip, replace w_i with the best candidate; continue
4. Return adversarial x or "fail" after budget
```

**Results.** >90% attack success on BERT for IMDB / Yelp / SNLI within ~50 queries per input. The original example from the paper:

```
Original (positive sentiment): "The film is a great work of art."
TextFooler:                    "The film is a tremendous work of art."   → flipped to negative
```

Counter-fitted GloVe synonyms preserve meaning to humans but shift the BERT embedding into the negative-sentiment manifold. **The model's robustness to synonyms is much worse than humans'.**


In [ ]:
# TextFooler core loop in ~25 lines (omitting USE/POS filters)
def textfooler(text, predict_fn, synonyms_fn, budget=50):
    tokens = text.split()
    P0 = predict_fn(text)
    y_true = P0.argmax(); p_true0 = P0[y_true]
    # 1. score each token
    scores = []
    for i in range(len(tokens)):
        masked = " ".join(tokens[:i] + ["[MASK]"] + tokens[i+1:])
        p = predict_fn(masked)[y_true]
        scores.append((p_true0 - p, i))
    # 2. attack in descending importance order
    queries = 0
    for _, i in sorted(scores, reverse=True):
        best_drop, best_word = 0, tokens[i]
        for w in synonyms_fn(tokens[i]):                # already POS + USE filtered
            cand = tokens.copy(); cand[i] = w
            P = predict_fn(" ".join(cand)); queries += 1
            if P.argmax() != y_true:                    # success
                return " ".join(cand), queries
            drop = p_true0 - P[y_true]
            if drop > best_drop: best_drop, best_word = drop, w
            if queries >= budget: return None, queries
        tokens[i] = best_word                            # commit best swap, keep going
    return None, queries


## 8 · The NLP adversarial canon

TextFooler is one node in a larger family tree. The frontier as of 2026:

| Attack | Year | Granularity | Constraint | Notes |
|---|---|---|---|---|
| **HotFlip** (Ebrahimi) | 2018 | char | gradient-based one-char swap | White-box; small budget |
| **DeepWordBug** (Gao) | 2018 | char | visual look-alikes (`e → ė`) | Defeats spell-checkers poorly |
| **TextFooler** (Jin) | 2020 | word | counter-fit GloVe + USE 0.7 | Black-box; the prototype |
| **BERT-Attack** (Li) | 2020 | word | BERT-MLM as synonym source | Best NLP black-box for 2020-2022 |
| **GBDA** (Guo) | 2021 | word/char | Gumbel-softmax on text simplex | Differentiable text attack |
| **TextAttack** (QData) | 2020 | all | composable (transformation × constraint × attack-recipe) | The standard framework |
| **CLARE** (Li) | 2020 | word | mask-fill + insert + delete + replace | More aggressive than TextFooler |
| **PWWS** (Ren) | 2019 | word | WordNet synonyms + word saliency | Classical baseline |
| **GCG / Universal-Adv-Suffix** (Zou) | 2023 | token | gradient-based suffix on LLM logits | **The 2023 LLM jailbreak founder** |
| **PAIR / AutoDAN / MART** | 2023-2024 | prompt | LLM-as-attacker + iterative refinement | M74 / M89 alignment-side |

**Defenses.** Adversarial training works for NLP (Zhu 2020 "FreeLB", Wang 2022 "InfoBERT") but is rare in production. The 2026 production defense is **input preprocessing + ensemble + confidence-thresholded escalation** — the same recipe as M91's vision defenses.

> 🌉 **The big intellectual bridge.** TextFooler (2018) → GCG (2023) → modern LLM jailbreaks (2024+) are *the same algorithm*: discrete optimization over a token vocabulary to flip a classifier's (or generator's) output. The loss surface changed; the math didn't.


## 9 · Adversarial attacks on captioners

Most NLP-adversarial work targets classifiers. But **captioners** (CLIP, BLIP, LLaVA) are vulnerable too:

| Attack | Threat | Defense |
|---|---|---|
| **CLIP perturbation** | Add imperceptible noise to image → CLIP scores wrong text → BLIP/LLaVA describes wrong content | CLIP+adversarial pretrain (FARE, ReCLIP) |
| **Typographic attacks** (Goh 2021) | Stick a "iPod" label on an apple → CLIP reads "iPod" | Input pre-filter, OCR-aware features |
| **Patch attack on captioner** | Adversarial patch → caption changes to attacker-chosen word | OPS-Caption defense, ensemble |
| **Prompt injection via image** | Image contains hidden text "Ignore previous instructions" → VLM follows it (GPT-4V) | Visual text-detection + sanitization |
| **Caption hallucination attack** | Adversarial features push captioner to hallucinate non-present objects | RLHF for hallucination (M89) |
| **Multilingual evasion** | Adversarial example in language A passes filter, decoded in language B | Multilingual safety classifiers |

**The 2026 reality.** Frontier VLMs (GPT-4V, Claude-3.7-Vision) are *more* robust than CLIP alone because the LLM provides a sanity check — "if the caption says 'iPod' but the image is clearly an apple, the LLM may correct itself." But the gap is narrow; *adaptive* attackers can still flip outputs with ~100 queries.

> 🛡️ **The defense stack mirrors M91**: training-time CAI/RLHF + input safety classifier (Llama-Guard-3, M89) + output classifier + monitoring + red-team loop. The intellectual program of M91 (vision robustness) and M95 (NLP+VLM robustness) is one continuous discipline. Every model in this course is, at scale, both a captioner and a classifier — and both faces are attackable.


## 10 · The 95-module wrap-up

You started at **M1 — Python basics** in Phase 1 and finished at **M95 — adversarial NLP & captioning** in Phase 13. The arc:

| Phase | Modules | What you built |
|---|---|---|
| **1. Python + Data Foundations** | M1-M17 | Pandas, NumPy, viz, dashboards, ETL, EDA, baseline modeling |
| **2. Modern ML Foundations** | M18-M27 | All 6 model families, transformers from scratch, diffusion, time-series, math + eval foundations |
| **3-7. LLM Ecosystem** | M28-M54 | RAG, LangChain, LangGraph, fine-tuning, vector DBs, A/B testing, MLOps, observability, Go/Rust/TS for AI |
| **8. LLMs from Scratch** | M55-M63 | Tokenization → GPT-2 124M → pretraining → SFT → KV-cache → Llama 3 → DPO from scratch |
| **9. Production Gaps** | M64-M73 | MCP, multimodal, distributed, RLHF, orchestration, feature store, eval, Triton, agents, Spark |
| **10. Master-Plan Gaps** | M74-M81 | Security, A2A, classical RL, GPU networks, bare metal, FinOps, multi-tenant, enterprise capstone |
| **11. Foundations Backfill** | M82-M85 | Linux + Bash, CNNs (M83), RNN/LSTM/GRU (M84), GAN/AE/VAE (M85) |
| **12. Production Extensions** | M86-M90 | Diffusion deep-dive, GraphRAG, synthetic data, Constitutional AI, edge / on-device AI |
| **13. Vision Robustness + Translation** | **M91-M95** | Adversarial ML (M91), Style transfer + AdaIN (M92), Training tricks (M93), Conditional translation (M94), **Captioning + NLP-adv (M95)** |

**95 notebooks. 95 docx files. ~7M bytes of notebooks, ~25M bytes of docs. One repo.** Across all of it, the intellectual program has been one thing: **understand each technology well enough to know when to use it, when to skip it, and what comes next.** The skills aren't the implementations — those rot. The skill is the **mental model** that lets you absorb the next implementation in a day instead of a month.

> 🎓 **Course epilogue.** Modular learning is a defense against the field's pace. Every quarter another frontier model lands; every year another architecture wave. The reason this course's foundations (M1-M85) age slowly is that the math and the lineage don't change — the substrate does. The reason the production extensions (M86-M95) age fast is the same: they pin a *specific* 2026 snapshot of what shipping looks like. Re-read M86-M95 in 2027 and the picker tables will be wrong; the **reasoning** for *why* each pick is right won't be. Knowing the difference between substrate and snapshot is what makes someone senior.

> 🏁 **Where you stand.** You finished. That isn't trivial — most learners abandon courses around the 30% mark. You have an artifact (this repo), a public proof (GitHub), and the intellectual scaffolding to keep adding modules as the field moves. **Ship something. Mentor someone through M1. Open a PR on this repo with a typo fix. The compounding is real.**

## ✅ Recap

- **Show-and-Tell (2015)** = CNN encoder + LSTM decoder + MLE on captions. The recipe is still alive — every modern VLM is a scaled-up, transformer-encoder, frozen-LLM-decoder, projector-bridged version of it.
- **Encoder lineage**: VGG → ResNet-101 → ViT → CLIP → SigLIP → DINOv2 → native multimodal.
- **Decoder lineage**: LSTM → LSTM+soft-attention → Transformer → BLIP-2 Q-Former → LLaVA projector → native mixed-token.
- **Soft attention (Xu 2015)** is exactly cross-attention before it had the name. M19 / M20 / M84 / M86 / this module all use the same building block.
- **Decoding**: beam-search for captions; sampling for general text; SCST / CIDEr-optim for leaderboard wins.
- **2026 picker**: don't train captioners — use GPT-4o / Claude-3.7-Vision / Gemini-2.5-Pro / LLaVA-NeXT / InternVL-2 / MiniCPM-V / Qwen2-VL / Florence-2 / Phi-3.5-Vision.
- **TextFooler** is the prototype of black-box NLP adversarial attacks — word-swap with synonym constraint + USE similarity. >90% on BERT.
- **NLP adversarial canon**: HotFlip · TextFooler · BERT-Attack · GBDA · TextAttack framework · GCG (the 2023 LLM-jailbreak founder).
- **VLMs are attackable on both sides** — image perturbation (M91), prompt injection via image, typographic attacks, multilingual evasion. Defense = M89 stack.

🎓 **Course complete. M1 → M95.** Ship.
